# E6 | Model XGBoost v2 — Tuned & Optimized
Consolidar feature selection + best hyperparameters em modelo de produção

## 📋 Objetivo

Criar versão **otimizada e pronta para produção** do XGBoost, consolidando:
1. **Feature Selection**: Top ~50-80 features por SHAP importance (87% redução)
2. **Hyperparameter Tuning**: Best params encontrados no grid search (max_depth=6, lr=0.1)
3. **Threshold Optimization**: Ajustar decisão threshold para F1-score máximo
4. **Explicit Encoding & Scaling**: Logs para reproducibilidade e serving em produção

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, pandas as pd, numpy as np
from datetime import datetime
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import xgboost as xgb
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, f1_score, auc
import shap
import matplotlib.pyplot as plt
import mlflow
import joblib

load_dotenv()
print('✅ Setup OK')

In [ ]:
RDS_HOST     = os.getenv('RDS_HOST')
RDS_PORT     = int(os.getenv('RDS_PORT', '5432'))
RDS_USER     = os.getenv('RDS_USER', 'postgres')
RDS_PASSWORD = os.getenv('RDS_PASSWORD')
RDS_DATABASE = os.getenv('RDS_DATABASE', 'aiops_gold')

engine = create_engine(f"postgresql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}")

query = '''
    SELECT * FROM gold_ml.ml_risk_dataset
    WHERE target_excedeu_tempo IS NOT NULL
'''
df = pd.read_sql(query, engine)

print(f'✅ Carregados {len(df)} incidentes')
print(f'   Positivos (risco): {(df.target_excedeu_tempo == 1).sum()}')
print(f'   Features: {df.shape[1] - 2}')

In [ ]:
feature_cols = [c for c in df.columns 
    if c not in ['incident_id', 'target_excedeu_tempo', 'duracao_horas', 'target_risco_sla']]
X = df[feature_cols].copy()
y = df['target_excedeu_tempo'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print(f'✅ Features após encoding: {X_train_prep.shape[1]}')

In [ ]:
mlflow.set_experiment('xgboost_v2_tuned')

with mlflow.start_run(run_name='xgboost_v2_final'):
    model_baseline = xgb.XGBClassifier(
        max_depth=6,
        learning_rate=0.1,
        n_estimators=100,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss',
        verbosity=0
    )
    
    model_baseline.fit(X_train_prep, y_train)
    y_pred_baseline = model_baseline.predict(X_test_prep)
    y_pred_proba_baseline = model_baseline.predict_proba(X_test_prep)[:, 1]
    
    auc_baseline = roc_auc_score(y_test, y_pred_proba_baseline)
    
    mlflow.log_params({
        'version': 'v2_tuned',
        'max_depth': 6,
        'learning_rate': 0.1,
        'n_estimators': 100,
        'n_features_total': X_train_prep.shape[1]
    })
    
    mlflow.log_metrics({
        'auc_roc_baseline': auc_baseline,
        'f1_baseline': f1_score(y_test, y_pred_baseline)
    })
    
    print(f'✅ XGBoost v2 (Baseline com tuned params)')
    print(f'   AUC-ROC: {auc_baseline:.4f}')
    print(f'   F1-Score: {f1_score(y_test, y_pred_baseline):.4f}')

In [ ]:
with mlflow.start_run(run_name='feature_selection_v2', nested=True):
    explainer = shap.TreeExplainer(model_baseline)
    shap_values = explainer.shap_values(X_test_prep)
    
    if isinstance(shap_values, list):
        shap_values_class1 = shap_values[1]
    else:
        shap_values_class1 = shap_values
    
    feature_importance = np.abs(shap_values_class1).mean(axis=0)
    feature_importance_sorted = np.argsort(feature_importance)[::-1]
    
    threshold = 0.005
    selected_features_mask = feature_importance >= threshold
    n_selected = selected_features_mask.sum()
    pct_reduction = 100 * (1 - n_selected / len(feature_importance))
    
    X_train_selected = X_train_prep[:, selected_features_mask]
    X_test_selected = X_test_prep[:, selected_features_mask]
    
    model_selected = xgb.XGBClassifier(
        max_depth=6,
        learning_rate=0.1,
        n_estimators=100,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss',
        verbosity=0
    )
    model_selected.fit(X_train_selected, y_train)
    y_pred_proba_selected = model_selected.predict_proba(X_test_selected)[:, 1]
    
    auc_selected = roc_auc_score(y_test, y_pred_proba_selected)
    auc_loss = auc_baseline - auc_selected
    
    mlflow.log_params({
        'shap_threshold': threshold,
        'n_features_selected': int(n_selected)
    })
    
    mlflow.log_metrics({
        'auc_roc_selected': auc_selected,
        'auc_loss': auc_loss,
        'pct_feature_reduction': pct_reduction
    })
    
    print(f'✅ Feature Selection')
    print(f'   Features selecionadas: {int(n_selected)} ({pct_reduction:.1f}% redução)')
    print(f'   AUC-ROC: {auc_selected:.4f} (perda: {auc_loss:.6f})')

In [ ]:
with mlflow.start_run(run_name='threshold_optimization_v2', nested=True):
    precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba_selected)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
    
    best_threshold_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_threshold_idx] if best_threshold_idx < len(thresholds) else 0.5
    best_f1 = f1_scores[best_threshold_idx]
    
    y_pred_optimized = (y_pred_proba_selected >= best_threshold).astype(int)
    
    from sklearn.metrics import precision_score, recall_score, confusion_matrix
    precision_opt = precision_score(y_test, y_pred_optimized)
    recall_opt = recall_score(y_test, y_pred_optimized)
    f1_opt = f1_score(y_test, y_pred_optimized)
    
    mlflow.log_params({'threshold_optimized': float(best_threshold)})
    mlflow.log_metrics({
        'f1_optimized': f1_opt,
        'precision_optimized': precision_opt,
        'recall_optimized': recall_opt
    })
    
    print(f'✅ Threshold Optimization')
    print(f'   Threshold ótimo: {best_threshold:.4f}')
    print(f'   F1-Score: {f1_opt:.4f} | Precision: {precision_opt:.4f} | Recall: {recall_opt:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr_baseline, tpr_baseline, _ = roc_curve(y_test, y_pred_proba_baseline)
roc_auc_baseline = auc(fpr_baseline, tpr_baseline)

fpr_selected, tpr_selected, _ = roc_curve(y_test, y_pred_proba_selected)
roc_auc_selected = auc(fpr_selected, tpr_selected)

axes[0].plot(fpr_baseline, tpr_baseline, label=f'Baseline (AUC={roc_auc_baseline:.4f})', linewidth=2)
axes[0].plot(fpr_selected, tpr_selected, label=f'Selected (AUC={roc_auc_selected:.4f})', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve Comparison')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(thresholds[:len(f1_scores)], f1_scores, linewidth=2, label='F1-Score')
axes[1].axvline(best_threshold, color='red', linestyle='--', label=f'Optimal threshold={best_threshold:.4f}')
axes[1].set_xlabel('Classification Threshold')
axes[1].set_ylabel('F1-Score')
axes[1].set_title('Threshold Optimization')
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig('xgboost_v2_evaluation.png', dpi=100, bbox_inches='tight')
mlflow.log_artifact('xgboost_v2_evaluation.png', 'evaluation')
plt.close()

print('✅ Gráficos de avaliação salvos')

In [ ]:
with mlflow.start_run(run_name='cross_validation_v2', nested=True):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    cv_scores = cross_val_score(
        model_selected,
        X_train_selected,
        y_train,
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1
    )
    
    mlflow.log_metrics({
        'cv_auc_mean': float(cv_scores.mean()),
        'cv_auc_std': float(cv_scores.std()),
        'cv_auc_fold_0': float(cv_scores[0]),
        'cv_auc_fold_1': float(cv_scores[1]),
        'cv_auc_fold_2': float(cv_scores[2]),
        'cv_auc_fold_3': float(cv_scores[3]),
        'cv_auc_fold_4': float(cv_scores[4])
    })
    
    print(f'✅ Cross-Validation')
    print(f'   AUC-ROC (CV): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    print(f'   Folds: {cv_scores}')

In [ ]:
from pathlib import Path

base_path = Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\xgboost_v2')
base_path.mkdir(parents=True, exist_ok=True)

joblib.dump(model_selected, str(base_path / 'xgboost_v2_model.pkl'))
joblib.dump(preprocessor, str(base_path / 'preprocessor.pkl'))
joblib.dump(selected_features_mask, str(base_path / 'feature_selection_mask.pkl'))

mlflow.log_artifact(str(base_path / 'xgboost_v2_model.pkl'), 'models')
mlflow.log_artifact(str(base_path / 'preprocessor.pkl'), 'preprocessing')
mlflow.log_artifact(str(base_path / 'feature_selection_mask.pkl'), 'preprocessing')

model_summary = pd.DataFrame({
    'metric': [
        'auc_roc_baseline',
        'auc_roc_selected_features',
        'f1_with_optimal_threshold',
        'precision_optimized',
        'recall_optimized',
        'cv_auc_mean',
        'cv_auc_std',
        'n_features_baseline',
        'n_features_selected',
        'optimal_threshold'
    ],
    'value': [
        auc_baseline,
        auc_selected,
        f1_opt,
        precision_opt,
        recall_opt,
        cv_scores.mean(),
        cv_scores.std(),
        X_train_prep.shape[1],
        int(n_selected),
        best_threshold
    ]
})
model_summary.to_csv(str(base_path / 'xgboost_v2_summary.csv'), index=False)

mlflow.set_tag('model_version', 'v2_tuned_production_ready')
mlflow.set_tag('feature_reduction_pct', f'{pct_reduction:.1f}%')
mlflow.set_tag('optimal_threshold', f'{best_threshold:.4f}')

print(f'✅ Salvos em {base_path}')
print(f'\n📊 Resumo final:')
print(model_summary.to_string(index=False))